In [92]:
import cv2
import mediapipe as mp

In [93]:
model_path = './face_landmarker.task'

In [94]:
BaseOptions =  mp.tasks.BaseOptions
FaceLandmarker =  mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions =  mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode =  mp.tasks.vision.RunningMode

# Create a face landmarker instance with the video mode:
options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1,
    min_face_presence_confidence=0.7)

In [95]:
LEFT_IRIS  = 468
RIGHT_IRIS = 473

In [96]:
def analyze_eye_contact(video_capture: cv2.VideoCapture, frame_step ):
    results = []

    with FaceLandmarker.create_from_options(options) as landmarker:
        frame_index = 0
        fps = video_capture.get(cv2.CAP_PROP_FPS) # reads the frame rate from the video file 30 or 60

        while True:
            _, frame = video_capture.read() # frame  the actual image data
            if frame is None:
                break

            if frame_index % frame_step == 0: # only every 5 frames
                h, w = frame.shape[:2] # saves the height and width of the frame
                timestamp_ms = int((frame_index / fps) * 1000) # saves the time of the frame

                # here cv2 reads the frame → fix colors → wrap it for MediaPipe.
                mp_image = mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                )

                detection = landmarker.detect_for_video(mp_image, timestamp_ms)

                if detection.face_landmarks:
                    lm    = detection.face_landmarks[0]
                    left  = (lm[LEFT_IRIS].x * w,  lm[LEFT_IRIS].y * h) # media return normalized coordinators (0-1)
                    right = (lm[RIGHT_IRIS].x * w, lm[RIGHT_IRIS].y * h) # so we multiply by width and height to get actual pixel positions
                else:
                    left = right = None

                results.append({
                    "frame": frame_index,
                    "left_iris": left,
                    "right_iris": right,
                })

            frame_index += 1

    return results

In [ ]:
cap = cv2.VideoCapture("C:\\Users\\beeko\\PycharmProjects\\AI-Video-interview-analysis-\\DataSet\\man1.mp4")  # 0 = webcam, or pass a file path
positions = analyze_eye_contact(cap, frame_step =10)
cap.release()
for p in positions:
    print(p)

{'frame': 0, 'left_iris': None, 'right_iris': None}
{'frame': 5, 'left_iris': (411.22606921195984, 125.8024549484253), 'right_iris': (464.7115273475647, 130.70302963256836)}
{'frame': 10, 'left_iris': (412.80819112062454, 125.82251071929932), 'right_iris': (466.48756182193756, 130.7151746749878)}
{'frame': 15, 'left_iris': (416.22984820604324, 126.45233631134033), 'right_iris': (470.74824261665344, 132.46766567230225)}
{'frame': 20, 'left_iris': (418.44079250097275, 126.83180809020996), 'right_iris': (472.9179050922394, 134.03001308441162)}
{'frame': 25, 'left_iris': (418.15736812353134, 130.41438102722168), 'right_iris': (472.139964222908, 137.25717544555664)}
{'frame': 30, 'left_iris': (415.7800239920616, 131.83198928833008), 'right_iris': (470.26181960105896, 137.9577398300171)}
{'frame': 35, 'left_iris': (414.77849447727203, 133.43313217163086), 'right_iris': (469.0495797395706, 138.72040271759033)}
{'frame': 40, 'left_iris': (415.76378613710403, 133.10272693634033), 'right_iris': 